In [8]:
from datetime import datetime
from dateutil.relativedelta import relativedelta

import pandas as pd
pd.set_option("display.max_columns",30)
import requests

In [ ]:
#---
#1.Get data from S3 (taxi, weather)
#2. weather data transformations ----DONE
#3. taxi data transformation ----DONE
#4. update dim_payment_type ----DONE
#5. update dim_company ----DONE
#6. update fact_taxi_trips with the ids from dim_payment_type and dim_company ----DONE
#7. upload dim_weather to S3
#8. upload fact_taxi_trips to S3
#9. upload dim_payment_type and dim_company (current and previous version)
#---


In [10]:
current_datetime=datetime.now() - relativedelta(months=2)
formatted_datetime=current_datetime.strftime("%Y-%m-%d")


url= (
    f"https://data.cityofchicago.org/api/v3/views/ajtu-isnz/query.json?app_token=ogjssL0qdXCJ16jANDq5lFDLc"
    f"&query=select * where trip_start_timestamp>='{formatted_datetime}T00:00:00' AND trip_start_timestamp<='{formatted_datetime}T23:59:59'"

)
response=requests.get(url)
data=response.json()

taxi_trips=pd.DataFrame(data)
taxi_trips.head()

,trip_id,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,pickup_community_area,dropoff_community_area,fare,tips,tolls,extras,trip_total,payment_type,company,pickup_centroid_latitude,pickup_centroid_longitude,pickup_centroid_location,dropoff_centroid_latitude,dropoff_centroid_longitude,dropoff_centroid_location,:id,:version,:created_at,:updated_at,pickup_census_tract,dropoff_census_tract
0,4a7b2ba9f018451c3437ccf6f283b021b782d431,21b6a618a3975334fec376b38bc8b8c88fb9f5f81c1789...,2025-09-27T23:45:00.000,2025-09-28T00:00:00.000,1135,4.81,6,28,30,7.62,0,0,38.12,Credit Card,Tac - American United Dispatch,41.944226601,-87.655998182,"{'type': 'Point', 'coordinates': [-87.65599818...",41.874005383,-87.66351755,"{'type': 'Point', 'coordinates': [-87.66351754...",row-tyuf-aisa~rinp,rv-6ivj-5zr3.3cvc,2025-10-09T18:15:56.782Z,2025-10-09T18:15:56.782Z,NaN,NaN
1,4a4c71ff9feff4d02bdae31b73e56377cde613ed,6a23e62ff9add07c054168d9bb0ee5c00d1a4306b3467c...,2025-09-27T23:45:00.000,2025-09-27T23:45:00.000,300,0.4,8,8,5,2,0,0,7,Credit Card,Transit Administrative Center Inc,41.892507781,-87.626214906,"{'type': 'Point', 'coordinates': [-87.62621490...",41.892042136,-87.63186395,"{'type': 'Point', 'coordinates': [-87.63186394...",row-jj2q~jscp~t35y,rv-z3w3_wbvg-9ay6,2025-10-09T18:15:56.782Z,2025-10-09T18:15:56.782Z,17031081500,17031081700
2,4a3980ae757a76317075d9dcf22eb17bddb4825f,202e21a8998abf8003539ebe81ca3033fc39b0d4569ec9...,2025-09-27T23:45:00.000,2025-09-28T00:00:00.000,1414,4.33,8,22,16.69,4.26,0,0,21.45,Mobile,5 Star Taxi,41.899602111,-87.633308037,"{'type': 'Point', 'coordinates': [-87.63330803...",41.92276062,-87.699155343,"{'type': 'Point', 'coordinates': [-87.69915534...",row-9fpt.i3p8~t7es,rv-uzhf_98wg_7jex,2025-10-09T18:15:56.782Z,2025-10-09T18:15:56.782Z,NaN,NaN
3,48330541c949d2bdb1a8edacc3dc3fc1588dd3aa,26689c424d89796e95dd65b0fc8ecb7ae34e3dc24a4984...,2025-09-27T23:45:00.000,2025-09-27T23:45:00.000,78,0.07,8,8,7.29,0,0,0,7.79,Mobile,City Service,41.892042136,-87.63186395,"{'type': 'Point', 'coordinates': [-87.63186394...",41.892042136,-87.63186395,"{'type': 'Point', 'coordinates': [-87.63186394...",row-5cvc-9bqm~kp9i,rv-d8er~k3wh-bk96,2025-10-09T18:15:56.782Z,2025-10-09T18:15:56.782Z,17031081700,17031081700
4,46a7d98734a8fe6c17dd2bfd4eff0afe2e877c87,a712fead8a31c26ab7948b116a760ac22af166d63d2fc2...,2025-09-27T23:45:00.000,2025-09-28T00:00:00.000,737,2.32,32,8,10,2,0,0,12.5,Credit Card,City Service,41.870607372,-87.622172937,"{'type': 'Point', 'coordinates': [-87.62217293...",41.899155613,-87.626210532,"{'type': 'Point', 'coordinates': [-87.62621053...",row-7wzu-jrpi~becf,rv-bqqt~fvj8.swh8,2025-10-09T18:15:56.782Z,2025-10-09T18:15:56.782Z,17031320600,17031081201


### Taxi data transformation

In [ ]:
def taxi_trips_transformations(taxi_trips: pd.DataFrame) -> pd.DataFrame:
    #---
    #performs transformation on taxi data

    #1. Drop selected columns
    #2. Drop NULL values across columns
    #3. Rename columns
    #4. Create datetime_for_weather helper column (for dim_weather join)
    #
    # :param taxi_trips: dataframe holding daily taxi trips
    #:return:           transformed taxi trips dataframe
    #---

    if not isinstance(taxi_trips, pd.DataFrame):                        #elemi hibakezelés a funkcióban
        raise TypeError("taxi_trips is a not valid pandas Dataframe")
    
    taxi_trips.drop(["pickup_census_tract","dropoff_census_tract","pickup_centroid_location",
                 "dropoff_centroid_location"],axis=1,inplace=True)
    taxi_trips.dropna(inplace=True)   #ha valamelyik rekordban Nan van akkor az egész rekordot törli (adattisztítás)

    taxi_trips.rename(columns={"pickup_community_area":"pickup_community_area_id",
                           "dropoff_community_area":"dropoff_community_area_id"},inplace=True)

    taxi_trips["trip_start_timestamp"]=pd.to_datetime(taxi_trips["trip_start_timestamp"])

    taxi_trips["datetime_for_weather"]=taxi_trips["trip_start_timestamp"].dt.floor("h")   

    return taxi_trips



In [ ]:

taxi_trips_transformed=taxi_trips_transformations(taxi_trips)

taxi_trips_transformed.head()

,trip_id,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,pickup_community_area_id,dropoff_community_area_id,fare,tips,tolls,extras,trip_total,payment_type,company,pickup_centroid_latitude,pickup_centroid_longitude,dropoff_centroid_latitude,dropoff_centroid_longitude,:id,:version,:created_at,:updated_at,datetime_for_weather
0,4a7b2ba9f018451c3437ccf6f283b021b782d431,21b6a618a3975334fec376b38bc8b8c88fb9f5f81c1789...,2025-09-27 23:45:00,2025-09-28T00:00:00.000,1135,4.81,6,28,30,7.62,0,0,38.12,Credit Card,Tac - American United Dispatch,41.944226601,-87.655998182,41.874005383,-87.66351755,row-tyuf-aisa~rinp,rv-6ivj-5zr3.3cvc,2025-10-09T18:15:56.782Z,2025-10-09T18:15:56.782Z,2025-09-27 23:00:00
1,4a4c71ff9feff4d02bdae31b73e56377cde613ed,6a23e62ff9add07c054168d9bb0ee5c00d1a4306b3467c...,2025-09-27 23:45:00,2025-09-27T23:45:00.000,300,0.4,8,8,5,2,0,0,7,Credit Card,Transit Administrative Center Inc,41.892507781,-87.626214906,41.892042136,-87.63186395,row-jj2q~jscp~t35y,rv-z3w3_wbvg-9ay6,2025-10-09T18:15:56.782Z,2025-10-09T18:15:56.782Z,2025-09-27 23:00:00
2,4a3980ae757a76317075d9dcf22eb17bddb4825f,202e21a8998abf8003539ebe81ca3033fc39b0d4569ec9...,2025-09-27 23:45:00,2025-09-28T00:00:00.000,1414,4.33,8,22,16.69,4.26,0,0,21.45,Mobile,5 Star Taxi,41.899602111,-87.633308037,41.92276062,-87.699155343,row-9fpt.i3p8~t7es,rv-uzhf_98wg_7jex,2025-10-09T18:15:56.782Z,2025-10-09T18:15:56.782Z,2025-09-27 23:00:00
3,48330541c949d2bdb1a8edacc3dc3fc1588dd3aa,26689c424d89796e95dd65b0fc8ecb7ae34e3dc24a4984...,2025-09-27 23:45:00,2025-09-27T23:45:00.000,78,0.07,8,8,7.29,0,0,0,7.79,Mobile,City Service,41.892042136,-87.63186395,41.892042136,-87.63186395,row-5cvc-9bqm~kp9i,rv-d8er~k3wh-bk96,2025-10-09T18:15:56.782Z,2025-10-09T18:15:56.782Z,2025-09-27 23:00:00
4,46a7d98734a8fe6c17dd2bfd4eff0afe2e877c87,a712fead8a31c26ab7948b116a760ac22af166d63d2fc2...,2025-09-27 23:45:00,2025-09-28T00:00:00.000,737,2.32,32,8,10,2,0,0,12.5,Credit Card,City Service,41.870607372,-87.622172937,41.899155613,-87.626210532,row-7wzu-jrpi~becf,rv-bqqt~fvj8.swh8,2025-10-09T18:15:56.782Z,2025-10-09T18:15:56.782Z,2025-09-27 23:00:00


In [13]:
taxi_trips_transformed.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16831 entries, 0 to 18351
Data columns (total 24 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   trip_id                     16831 non-null  object        
 1   taxi_id                     16831 non-null  object        
 2   trip_start_timestamp        16831 non-null  datetime64[ns]
 3   trip_end_timestamp          16831 non-null  object        
 4   trip_seconds                16831 non-null  object        
 5   trip_miles                  16831 non-null  object        
 6   pickup_community_area_id    16831 non-null  object        
 7   dropoff_community_area_id   16831 non-null  object        
 8   fare                        16831 non-null  object        
 9   tips                        16831 non-null  object        
 10  tolls                       16831 non-null  object        
 11  extras                      16831 non-null  object        


### Dim company and dim payment type update

In [15]:
def update_dim_company_dim_payment_type(taxi_trips: pd.DataFrame, dim_df: pd.DataFrame, id_col: str, value_col: str) -> pd.DataFrame:
    #---Extend the dimenion dataframe with new value if has any (generic)
    #:param taxi_trips:         dataframe daily taxi trips
    #:param dim_df:             dataframe dimension data (company, payment)
    #:param id_col:             id columns of dimension dataframe
    #:param value_col           name of column of dimension dataframe contining values
    #:return: extended dimension data
    #---
    todays_dim_data=pd.DataFrame(taxi_trips[value_col].unique(),columns=[value_col])

    new_dim_data=todays_dim_data[~todays_dim_data[value_col].isin(dim_df[value_col])]

    if not new_dim_data.empty:
        max_id=dim_df[id_col].max()
        new_dim_data[id_col]=range(max_id+1,max_id+1+len(new_dim_data))
        dim_df=pd.concat([dim_df,new_dim_data],ignore_index=True)

    return dim_df







In [16]:
dim_payment_type=taxi_trips["payment_type"].drop_duplicates().reset_index(drop=True)
dim_payment_type=pd.DataFrame(
    {"payment_type_id":range(1,len(dim_payment_type)+1),
     "payment_type":dim_payment_type
    }
)

dim_company=taxi_trips["company"].drop_duplicates().reset_index(drop=True)
dim_company=pd.DataFrame(
    {"company_id":range(1,len(dim_company)+1),
     "company":dim_company
    }
)


In [20]:
dim_payment_type_updated=update_dim_company_dim_payment_type(taxi_trips, dim_payment_type, "payment_type_id","payment_type")
dim_company_updated=update_dim_company_dim_payment_type(taxi_trips, dim_company, "company_id","company")

In [21]:
dim_payment_type_updated

,payment_type_id,payment_type
0,1,Credit Card
1,2,Mobile
2,3,Cash
3,4,Unknown
4,5,Prcard
5,6,No Charge
6,7,Dispute


In [22]:
dim_company_updated

,company_id,company
0,1,Tac - American United Dispatch
1,2,Transit Administrative Center Inc
2,3,5 Star Taxi
3,4,City Service
4,5,Flash Cab
5,6,Taxicab Insurance Agency Llc
6,7,Medallion Leasin
7,8,Sun Taxi
8,9,Blue Ribbon Taxi Association
9,10,Chicago City Taxi Association


### update fact_taxi_trips with company and payment_type ids

In [23]:
def update_fact_taxi_trips_with_dimension_data(taxi_trips: pd.DataFrame, dim_payment_type: pd.DataFrame,dim_company: pd.DataFrame) ->pd.DataFrame:
    #--Update fact_taxi_trips Dataframe with the dim_company and dim_payment_type ids and delete the string columns (generic)
    #:param taxi_trips:         dataframe daily taxi trips
    #:param dim_payment_type:    payment type master table
    #:param dim_company:         company master table
    #:return: taxi trips data with ids without company and payment type values
    #---
    fact_taxi_trips=taxi_trips.merge(dim_payment_type,on="payment_type")
    fact_taxi_trips=fact_taxi_trips.merge(dim_company,on="company")
    fact_taxi_trips.drop(["payment_type","company"],axis=1,inplace=True)

    return fact_taxi_trips


In [24]:
taxi_trips_transformed_with_dim_ids = update_fact_taxi_trips_with_dimension_data(taxi_trips_transformed, dim_payment_type, dim_company)

taxi_trips_transformed_with_dim_ids.head()

,trip_id,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,pickup_community_area_id,dropoff_community_area_id,fare,tips,tolls,extras,trip_total,pickup_centroid_latitude,pickup_centroid_longitude,dropoff_centroid_latitude,dropoff_centroid_longitude,:id,:version,:created_at,:updated_at,datetime_for_weather,payment_type_id,company_id
0,4a7b2ba9f018451c3437ccf6f283b021b782d431,21b6a618a3975334fec376b38bc8b8c88fb9f5f81c1789...,2025-09-27 23:45:00,2025-09-28T00:00:00.000,1135,4.81,6,28,30,7.62,0,0,38.12,41.944226601,-87.655998182,41.874005383,-87.66351755,row-tyuf-aisa~rinp,rv-6ivj-5zr3.3cvc,2025-10-09T18:15:56.782Z,2025-10-09T18:15:56.782Z,2025-09-27 23:00:00,1,1
1,4a4c71ff9feff4d02bdae31b73e56377cde613ed,6a23e62ff9add07c054168d9bb0ee5c00d1a4306b3467c...,2025-09-27 23:45:00,2025-09-27T23:45:00.000,300,0.4,8,8,5,2,0,0,7,41.892507781,-87.626214906,41.892042136,-87.63186395,row-jj2q~jscp~t35y,rv-z3w3_wbvg-9ay6,2025-10-09T18:15:56.782Z,2025-10-09T18:15:56.782Z,2025-09-27 23:00:00,1,2
2,4a3980ae757a76317075d9dcf22eb17bddb4825f,202e21a8998abf8003539ebe81ca3033fc39b0d4569ec9...,2025-09-27 23:45:00,2025-09-28T00:00:00.000,1414,4.33,8,22,16.69,4.26,0,0,21.45,41.899602111,-87.633308037,41.92276062,-87.699155343,row-9fpt.i3p8~t7es,rv-uzhf_98wg_7jex,2025-10-09T18:15:56.782Z,2025-10-09T18:15:56.782Z,2025-09-27 23:00:00,2,3
3,48330541c949d2bdb1a8edacc3dc3fc1588dd3aa,26689c424d89796e95dd65b0fc8ecb7ae34e3dc24a4984...,2025-09-27 23:45:00,2025-09-27T23:45:00.000,78,0.07,8,8,7.29,0,0,0,7.79,41.892042136,-87.63186395,41.892042136,-87.63186395,row-5cvc-9bqm~kp9i,rv-d8er~k3wh-bk96,2025-10-09T18:15:56.782Z,2025-10-09T18:15:56.782Z,2025-09-27 23:00:00,2,4
4,46a7d98734a8fe6c17dd2bfd4eff0afe2e877c87,a712fead8a31c26ab7948b116a760ac22af166d63d2fc2...,2025-09-27 23:45:00,2025-09-28T00:00:00.000,737,2.32,32,8,10,2,0,0,12.5,41.870607372,-87.622172937,41.899155613,-87.626210532,row-7wzu-jrpi~becf,rv-bqqt~fvj8.swh8,2025-10-09T18:15:56.782Z,2025-10-09T18:15:56.782Z,2025-09-27 23:00:00,1,4
